# Урок 10 - ИИ агенти в продукционна среда

В този урок ще научите **продукционни шаблони** за ИИ агенти с използване на Microsoft Agent Framework (Python). Обхващаме:

- **Наблюдаемост** — добавяне на времеви измервания и логване към взаимодействията на агента
- **Оценяване** — използване на агент-оценител за оценяване на качеството на отговорите
- **Управление на разходите** — стратегии за оптимизация на токените и избор на модел

Сценарият е един **туристически агент**, който помага на потребителите да планират пътувания, с добавен мониторинг и оценяване.


## Настройка


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Съображения при внедряване в продукция

Преминаването на AI агенти от прототипи към продукция изисква внимателно отношение към трите стълба:

1. **Наблюдаемост** — Трябва да имате видимост какво прави агентът, колко време отнема и кои инструменти извиква. Без проследяване и регистриране, отстраняването на грешки в продукция е почти невъзможно.

2. **Оценка** — Автоматизираните проверки за качество гарантират, че отговорите на агента остават точни, пълни и полезни с течение на времето. Оценяващ агент може да оценява отговорите спрямо дефинирани критерии.

3. **Управление на разходите** — Използването на токени влияе директно на разходите. Стратегии като оптимизация на подсказките, избор на модел и кеширане помагат да се поддържат разходите под контрол, без да се жертва качеството.


## Изграждане на наблюдаем агент

Дефинираме инструменти за пътуване и опаковаме извикването на агента с измерване на времето, за да можем да наблюдаваме латентността. В производствена среда бихте интегрирали с OpenTelemetry или подобен бекенд за трасировки.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Шаблони за оценяване

Често използван производствен модел е да се използва втори агент като **оценител**. Оценителят дава оценка на отговора на основния агент спрямо предварително зададени критерии като пълнота, точност и полезност.

Това позволява:
- Автоматизирани контроли за качество преди отговорите да достигнат до потребителите
- Откриване на регресия при промяна на подсказките или моделите
- Непрекъснато наблюдение на представянето на агента с течение на времето


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегии за управление на разходите

Контролирането на разходите е критично за производствените AI агенти. Ето ключови стратегии:

| Стратегия | Описание |
|---|---|
| **Оптимизация на подсказките** | Дръжте системните инструкции кратки. Премахвайте излишния контекст, за да намалите броя входни токени. |
| **Избор на модел** | Използвайте по-малки, по-евтини модели (например GPT-4o-mini) за прости задачи като класификация или извличане, а по-големите модели запазете за сложни разсъждения. |
| **Кеширане** | Кеширайте резултатите от инструменти и често задавани заявки, за да избегнете излишни API повиквания. |
| **Бюджети за токени** | Задайте ограничения на `max_tokens`, за да предотвратите неочаквано дълги отговори. |
| **Групиране** | Групирайте няколко потребителски заявки в едно API повикване, когато е възможно. |

На практика, многостепенният подход работи добре: насочвайте простите заявки към бърз и евтин модел и пренасочвайте само сложните запитвания към по-способен.


## Обобщение

В този урок научихте как да:

1. **Добавяне на наблюдаемост** към взаимодействията на агента чрез измерване на времето и записване на логове, полагайки основите за проследяване и мониторинг.
2. **Оценяване на отговорите на агента** автоматично чрез използване на оценяващ агент, който дава оценка за пълнота, точност и полезност.
3. **Управление на разходите** чрез оптимизация на подсказките, избор на модел, кеширане и бюджети за токени.

Тези производствени модели помагат да се гарантира, че вашите ИИ агенти са надеждни, измерими и икономически ефективни в мащаб.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводаческа услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на езика, на който е написан, трябва да се смята за авторитетен източник. За критична информация се препоръчва професионален превод от човешки преводач. Не носим отговорност за каквито и да е недоразумения или погрешни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
